<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Convolution and patches, in one lesson

**[Convolution and patches, in one lesson](https://learning.nextia-ai.com/courses/vision/m02/convolution-and-patches-in-one-lesson/)** · Vision AI: See, Count and Inspect · Module 2, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** slide a small filter over an image by hand, see what an edge filter responds to in a real photo, shrink the result with pooling, and cut a photo into the patches that a vision transformer reads.

| | |
|---|---|
| **Time** | About 20 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | HOG and edge directions, from [Hand-made features](https://learning.nextia-ai.com/courses/vision/m02/hand-made-features/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/convolution-and-patches-in-one-lesson/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M02-L03-convolution-and-patches/convolution-and-patches-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 15 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "compare.py": "65e58d2fe043105e3ff09626861a8f64e5548039df760ea10d6fc02c71f8fec3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A filter on a tiny image

The grid is a grey image of 6 × 6 pixels: dark (10) on the left, bright (200) on the right. The filter `vertical` subtracts the left column of a 3 × 3 window from its right column.

> **Predict.** Where in the output will the numbers be large: on the left, in the middle, or on the right?

In [ ]:
def conv2d(image, kernel):
    """Slide the filter over the image; at each place, multiply the numbers that overlap and add them up."""
    kh, kw = kernel.shape
    out = np.zeros((image.shape[0] - kh + 1, image.shape[1] - kw + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = (image[i:i + kh, j:j + kw] * kernel).sum()
    return out

grid = np.array([[10, 10, 10, 200, 200, 200]] * 4 + [[10, 10, 10, 10, 200, 200]] * 2)
vertical = np.array([[-1, 0, 1], [-1, 0, 1], [-1, 0, 1]])

print(conv2d(grid, vertical).astype(int))

**Check.** You should see this output, the same as on the lesson page:

```text
[[  0 570 570   0]
 [  0 570 570   0]
 [  0 380 570 190]
 [  0 190 570 380]]
```

The large numbers (570) are where the window covers the edge. Where the window sees only one colour, the output is 0.

## 2. The same filter on a real photo

The photo is a wire chair on a terrace (a catalogue test photo). We use NumPy's sliding windows, which do the same as `conv2d`, only faster.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def conv_fast(image, kernel):
    windows = sliding_window_view(image, kernel.shape)
    return (windows * kernel).sum(axis=(-2, -1))

photo = Image.open("data/catalogue/test/chair/P-434b4e4b.jpg").convert("L").resize((224, 224))
g = np.asarray(photo, float) / 255
v = np.abs(conv_fast(g, vertical))
h = np.abs(conv_fast(g, vertical.T))
print(g.shape, v.shape)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, a, t in zip(axes, (g, v, h), ("photo, grey", "vertical filter", "horizontal filter")):
    ax.imshow(a, cmap="gray" if t.startswith("photo") else "gray_r"); ax.set_title(t); ax.axis("off")
plt.show()

> **Check.** The shapes are `(224, 224) (222, 222)`: a 3 × 3 window fits 222 times along each side. The vertical filter shows the chair's legs and the upright wires; the horizontal filter shows the edge of the terrace and the flat wires.

The photo is by Timothy Brown (CC BY 2.0), from Open Images.

## 3. Pooling: keep the strongest answer

**Max pooling** keeps the largest number in each 2 × 2 block. The map gets 4 times smaller, and a small shift of the photo changes it less.

In [ ]:
def max_pool(a, size=2):
    h, w = a.shape[0] // size * size, a.shape[1] // size * size
    return a[:h, :w].reshape(h // size, size, w // size, size).max(axis=(1, 3))

small = max_pool(v)
print(v.shape, "->", small.shape)

## 4. Patches: what a vision transformer reads

A **vision transformer** (ViT) does not slide a filter. It cuts the photo into squares of 16 × 16 pixels, called **patches**, and reads them as a sequence, like the words of a sentence.

In [ ]:
rgb = np.asarray(Image.open("data/catalogue/test/mug/P-ab240c16.jpg").convert("RGB").resize((224, 224)))
patches = rgb.reshape(14, 16, 14, 16, 3).transpose(0, 2, 1, 3, 4).reshape(196, 16 * 16 * 3)
print(patches.shape)
fig, axes = plt.subplots(1, 8, figsize=(8, 1.2))
for j, ax in enumerate(axes):
    ax.imshow(rgb[6 * 16:7 * 16, j * 16:(j + 1) * 16]); ax.axis("off")
plt.show()

> **Check.** You should see `(196, 768)`: 14 × 14 = 196 patches, each with 16 × 16 pixels × 3 channels = 768 numbers. The pictures are the first 8 patches of row 7, through the mug. The photo is by Joris de Vrede (CC BY 2.0).

## 5. Your turn: two small changes

> **Your turn.**
> 1. Write the **horizontal** edge filter `horizontal`: it subtracts the top row of a 3 × 3 window from its bottom row. Put `conv2d(grid, horizontal)` in `out_h`.
> 2. A model cuts a 224 × 224 photo into patches of 32 × 32 pixels instead. Put the number of patches in `n_patches_32`.

In [ ]:
horizontal = ...
out_h = ...
n_patches_32 = ...

In [ ]:
expect_hash('out_h', out_h, '250f54ff1c67ff44')

In [ ]:
expect('n_patches_32', n_patches_32, 49)

Look at `out_h`: where is it not zero? Only where the edge between dark and bright changes column, in the lower rows. A horizontal filter answers to a step from top to bottom, and the grid has one only there.

## Next

You slid a filter by hand and cut a photo into patches. The [lesson page](https://learning.nextia-ai.com/courses/vision/m02/convolution-and-patches-in-one-lesson/) shows what a trained network's first filters look like, and has the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Transfer learning](https://learning.nextia-ai.com/courses/vision/m02/transfer-learning/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m02/convolution-and-patches-in-one-lesson/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.